In [1]:
# Extending Built-in Object Types
    # customizing built-in types
    # classes vs. types
    # __slots__ and properties
    # static methods and class methods
    # decorators and metaclasses
    # the complete super() mechanism
    # common OOP “gotchas

    # Embedding/delegation — put a built-in object such as a list inside your class and delegate operations to it.
    # Inheritance — directly subclass a built-in type such as list

# Extending Types by Embedding
from setwrapper import Set
x = Set([1, 3, 5, 7, 3])
x.union(Set([1, 4, 7]))
x | Set([1, 4, 6, 4])

# Extending Types by Subclassing
    # Instead of embedding a built-in type inside your class, you can inherit directly from it.

    # | Embedding                    | Subclassing                                 |
    # | ---------------------------- | ------------------------------------------- |
    # | Class **contains** a list    | Class **is a** list                         |
    # | `self.data` stores the list  | `self` is the list                          |
    # | You must delegate operations | Most operations are inherited automatically |
    # | More code                    | Usually shorter                             |

%run typesubclass.py
%run setsubclass.py
    # Subclassing = take an existing built-in type and customize or extend it, while keeping its existing behavior.

# The Python Object Model
    # In Python, classes are types, and classes themselves are objects.

# Classes Are Types Are Classes
class Hack: pass                    # A humble user-defined class
I = Hack()                          # Make an instance by calling the class
type(I)                             # Type is the user-defined class of origin
type(Hack)                          # User-defined classes are instances of type
I.__class__, Hack.__class__
isinstance(I, object), isinstance(Hack, object)
I = 'hack'                          # Make an instance by literal syntax, or str()
type(I)                             # Built-in objects are instances of classes
type(str)                           # Built-in classes are instances of type
I.__class__, str.__class__
isinstance(I, object), isinstance(str, object)

type(type)                          # The type class ends the chain
type(type(type))                    # Hmm...the top of the chain

# Some Instances Are More Equal Than Others
    # Python has two kinds of instances: class objects and ordinary objects. Both are objects, but they behave differently

class C:                    # C is itself a class object, created by the metaclass type.
    pass
I = C()                     # I is an ordinary instance created from class C

print(isinstance(I, type))  # False
print(isinstance(C, type))  # True

    # A metaclass is a class that creates classes. Python's default metaclass is type.
    # Here, "type" creates class C, and calling C() creates instance I.

# The Inheritance Bifurcation
    # Python has two inheritance search paths: one for ordinary instances and another, additional path for class objects.

# 1. Ordinary instance inheritance
isinstance(I, C), type(I)
I, I.__class__, I.__class__.__bases__
C, C.__bases__, C.__class__, C.__class__.__bases__
# The search path is: I → C → object

# 2. Class inheritance has an additional search
I, I.__class__.__mro__
C.__mro__, C.__class__.__mro__
[I.__dict__] + [x.__dict__ for x in I.__class__.__mro__]
[x.__dict__ for x in C.__mro__] + [x.__dict__ for x in C.__class__.__mro__]

# So the class-object search involves two MRO sequences: C → object, then type → object as the additional search path.

# 3. Why does C.__bases__ work but I.__bases__ fail?
print(C.__bases__)  # (<class 'object'>,)
print(I.__bases__)  # AttributeError

'__bases__' in I.__dict__           # Not in instance
'__bases__' in C.__dict__           # Not in instance's class
'__bases__' in C.__class__.__dict__ # In instance's class's class
C.__bases__ # Instance does not inherit!
I.__bases__
# __bases__ is a class-related attribute available through the class/metaclass attribute lookup mechanisms.

# The Metaclass/Class Dichotomy
    # Python unified classes and types, but this creates some complexity. The author questions whether this unification is worth the extra complications.
# What is the dichotomy?
# Class: creates ordinary instances, such as A() or MyClass().
# Metaclass: creates class objects; Python uses type by default.
type(object)                                 # Fetch object type
type(classname, superclasses, attributedict) # Make a class/type

# 1. type() has two roles
class A:
    pass
x = A()
print(type(x))  # <class '__main__.A'>

# Role 1 — inspect an object's type: pass one argument.
type(x)  # Returns the object's type
# Role 2 — create a class dynamically: pass three arguments.
MyClass = type(
    "MyClass",       # Class name
    (object,),       # Parent classes
    {"x": 10}        # Class attributes
)
print(MyClass.x)  # 10

# And One “object” to Rule Them All
    # Every Python class ultimately inherits from the built-in object class, directly or indirectly.
# 1. object is inherited automatically
class C:
    pass
class D(object):
    pass
print(C.__bases__)  # (<class 'object'>,)
print(D.__bases__)  # (<class 'object'>,)

# 2. object and type have a circular relationship
print(type(C))       # <class 'type'>
print(type(object))  # <class 'type'>
print(type(type))    # <class 'type'>

print(type.__bases__)    # (<class 'object'>,)
print(object.__bases__)  # ()
# type inherits from object.
# object is an instance of type.
# type is also an instance of itself.

# 3. Why is object useful?
# Every class inherits default behavior from object, including methods such as __repr__()
class C:
    pass
x = C()
print(repr(x))  # Default object representation

class C: pass
class D(object): pass
dir(C) == dir(D)
C.__bases__, D.__bases__

type is object
type(type), type(object)
isinstance(type, object), isinstance(object, type)
type.__bases__, object.__bases__

class C: pass   # All classes inherit object defaults
X = C()
X.__repr__

# Advanced Attribute Tools
    # __slots__ declares which instance attributes a class can have.
    # It can prevent accidental attributes and reduce memory usage when creating many instances.

# Slots: Attribute Declarations
# Slot basics
# 1. Declare allowed attributes
class Limiter:
    __slots__ = ['age', 'name', 'job']
person = Limiter()
person.age = 40
person.name = "Alex"
print(person.age)  # 40
person.ape = 1000  # AttributeError

# 2. Attributes must still be assigned before use
person = Limiter()
print(person.age)  # AttributeError: not assigned yet
person.age = 40
print(person.age)  # 40

# 3. Why use __slots__?
# Catch typos: accidental attribute names are rejected.
# Save memory: instances can avoid storing attributes in a separate __dict__.
# Potentially improve speed: speed benefits may be trivial in modern Python.

# You shouldn’t normally use slots
    # Avoid __slots__ unless you have a clear reason, especially a need to save memory when creating a huge number of objects.

# Slots and namespace dictionaries
    # Normally, Python stores an object's instance attributes in __dict__. With __slots__,
    # attributes can be stored without a dictionary, which saves memory but changes how you access and manage them.

# 1. Slots normally remove __dict__
class Person:
    __slots__ = ['name', 'age']
p = Person()
p.name = "Alex"
p.age = 40
print(p.name)       # Alex
print(p.__dict__)   # AttributeError

# 2. getattr(), setattr(), and dir() still work
print(getattr(p, 'name'))  # Alex
setattr(p, 'age', 41)
print(p.age)               # 41
print('name' in dir(p))    # True

# 3. Add __dict__ to allow extra attributes
class Person:
    __slots__ = ['name', '__dict__']
p = Person()
p.name = "Alex"
p.city = "Nukus"    # Allowed: stored in __dict__
print(p.name)       # Alex
print(p.__dict__)   # {'city': 'Nukus'}

# 4. Why listing attributes is tricky
for attr in list(getattr(p, '__dict__', [])) + getattr(p, '__slots__', []):
    print(attr, '=>', getattr(p, attr))


class C:
    __slots__ = ['a', 'b']  # __slots__ means no __dict__ by default

I = C()
I.a = 1
I.a
I.__dict__

getattr(I, 'a')
setattr(I, 'b', 2)      # But getattr() and setattr() still work
I.b
'a' in dir(I)           # And dir() finds slot attributes too
'b' in dir(I)           # Though __dict__ access will fail
I.__dict__

class D:
    __slots__ = ['a', 'b']
    def __init__(self):
        self.d = 4      # Cannot add new names if no __dict__
I = D()

class D:
    __slots__ = ['a', 'b', '__dict__']  # Name __dict__ to include one too
    c = 3                               # Class attrs work normally
    def __init__(self):
        self.d = 4                      # d stored in __dict__, a is a slot
I = D()
I.d
I.c
I.a                                     # All instance attrs undefined until assigned
I.a = 1
I.b = 2

I.__dict__                  # Some objects have both __dict__ and slot names
                            # getattr() can fetch either type of attr
I.__slots__

getattr(I, 'a'), getattr(I, 'c'), getattr(I, 'd')   # Fetches all 3 forms

for attr in list(I.__dict__) + I.__slots__:         # Wrong...
    print(attr, '=>', getattr(I, attr))

for attr in list(getattr(I, '__dict__', [])) + getattr(I, '__slots__', []):
    print(attr, '=>', getattr(I, attr))

# Multiple __slot__ lists in superclasses
    # The main idea is that __slots__ are not automatically combined into one list through inheritance,
    # even though an instance can use slots from multiple classes

class E:
    __slots__ = ['c', 'd']          # Superclass has slots

class D(E):
    __slots__ = ['a', '__dict__']   # But so does its subclass

I = D()                             # The instance gets the union of each
dir(I)
I.a = 1; I.b = 2; I.c = 3           # slots: a, c, __dict__: b
I.a, I.c

# Inspecting just the inherited slots list won’t pick up slots defined higher in a class tree:
E.__slots__                         # But __slots__ not concatenated
D.__slots__
I.__slots__                         # Instance inherits *lowest* __slots__
I.__dict__                          # And has its own attr dict

for attr in list(getattr(I, '__dict__', [])) + getattr(I, '__slots__', []):
    print(attr, '=>', getattr(I, attr))
b => 2                              # Other superclass slots missed!
a => 1
dir(I)                              # But dir() includes all slot names

# Handling slots and other “virtual” attributes generically
    # __dict__ alone cannot show all an object's attributes. Use dir() to find attribute names and getattr() to retrieve their values.
class Slotful:
    __slots__ = ['a', 'b', '__dict__']
    def __init__(self, data):
        self.c = data

I = Slotful(3)
I.a, I.b = 1, 2
I.a, I.b, I.c                               # Normal attribute fetch
I.__dict__                                  # Both __dict__ and slots storage
[x for x in dir(I) if not x.startswith('__')]
I.__dict__['c']                             # __dict__ is only one attr source
getattr(I, 'c'), getattr(I, 'a')            # dir+getattr is broader than __dict__
                                            # applies to slots, properties, descrip
for a in (x for x in dir(I) if not x.startswith('__')):
    print(a, '=>', getattr(I, a))

# Slot usage rules
    # __slots__ saves memory by avoiding an instance __dict__, but to work effectively, it requires careful design across the inheritance hierarchy.
    # 1. The six rules of __slots__
    # Rule	Explanation
    # 1. Subclass has slots, superclass doesn't	The inherited instance dictionary still exists, so memory savings are limited.
    # 2. Superclass has slots, subclass doesn't	The subclass gets an instance dictionary, reducing the memory benefit.
    # 3. Duplicate slot names	A subclass redefining a parent's slot hides the parent's slot descriptor. Avoid this.
    # 4. No class-level defaults	You cannot define a class attribute with the same name as a slot.
    # 5. Multiple inheritance	Two parent classes with nonempty slots generally cause a layout conflict.
    # 6. No __dict__ or weak references by default	Slots restrict arbitrary attributes and weak references unless explicitly enabled.

class C: pass                   # Bullet 1: slots in sub but not super
class D(C): __slots__ = ['a']   # Makes instance dict for non-slots
I = D()                         # But slot name still managed in class
I.a = 1; I.b = 2
I.__dict__
D.__dict__.keys()

class C: __slots__ = ['a']      # Bullet 2: slots in super but not sub
class D(C): pass                # Makes instance dict for non-slots
I = D()                         # But slot name still managed in class
I.a = 1; I.b = 2
I.__dict__
C.__dict__.keys()

class C: __slots__ = ['a']      # Bullet 3: only lowest slot accessible
class D(C): __slots__ = ['a']   # Superclass slot 'a' is pointless

class C: __slots__ = ['a']; a = 99 # Bullet 4: no class-level defaults

class C: __slots__ = ['a']         # Bullet 5: only one nonempty in mixins
class D: __slots__ = ['a']         # Use empty slots or omit in all but one
class E(C, D): pass

# Example impacts of slots: ListTree and mapattrs
    # a class without __slots__, such as ListTree, can preserve an instance's __dict__
    # even when mixed with classes that use slots. This can make tools work,
    # but it may reduce the memory-saving benefit of slots.

class C(ListTree): pass
I = C()                                   # OK: no __slots__ used
print(I)
class C(ListTree): __slots__ = ['a', 'b'] # OK: superclass produces __dict__
I = C()
I.c = 3
print(I)                                  # Displays c at I, a and b at C

# Multiple inheritance
class A: __slots__ = ['a']                # Both OK by bullet 1 above
class B(A, ListTree): pass
print(B())
class A: __slots__ = ['a']
class B(A, ListTree): __slots__ = ['b']   # Displays b at B, a at A
print(B())

# Unsafe approach:
attrs = obj.__dict__
# Safer approach:
attrs = getattr(obj, '__dict__', {})

# To inspect a broader range of attributes:
for name in dir(obj):
    if not name.startswith('__'):
        try:
            print(name, getattr(obj, name))
        except Exception:
            pass

# What about slots speed?
    # __slots__ usually saves memory, but its speed advantage is small and depends on the workload and Python implementation.
%run slots-test.py

# Properties: Attribute Accessors
    # a property allows you to use a method like an ordinary attribute.
    # @property — controls reading an attribute.
    # @age.setter — controls assignment and can validate values.
    # __getattr__ — runs when an attribute isn't found normally.
    # __setattr__ — intercepts every attribute assignment.
    # __slots__ — controls attribute storage and can reduce memory usage

    # Properties control how attributes behave; slots control how attributes are stored.

# Property basics
    # A property lets a class control how an attribute is read, assigned, or deleted, without changing the usual syntax obj.age


class WithOperators:
    def __getattr__(self, name): # On undefined attr
        if name == 'age':
            return 40
        else:
            raise AttributeError(name)
x = WithOperators()
x.age                           # Runs __getattr__
x.name                          # Runs __getattr__

# 1. Creating a property
class WithProperties:
    def getage(self):
        return 40
    age = property(getage)  # (get?, set?, del?, docs?), or @
x = WithProperties()
x.age                       # Runs getage
x.name                      # Normal fetch

# 2. Adding a setter
class WithProperties:
    def getage(self):
        print('get age')
        return 40
    def setage(self, value):
        print('set age:', value)
        self._age = value
    age = property(getage, setage)

x = WithProperties()
x.age                   # Runs getage
x.age = 42              # Runs setage
set age: 42
x._age                  # Normal fetch: no getage call
x.job = 'hacker'        # Normal assign: no setage call
x.job                   # Normal fetch: no getage call

class WithOperators:
    def __getattr__(self, name):           # On undefined attr
        if name == 'age':
            print('get age')
            return 40
        else:
            raise AttributeError(name)
    def __setattr__(self, name, value):    # On all assignments
        print('set:', name, value)
        if name == 'age':
            self.__dict__['_age'] = value  # Or object.__setattr__(self, ...)
        else:
            self.__dict__[name] = value
x = WithOperators()
x.age                           # Runs __getattr__
x.age = 41                      # Runs __setattr__
x._age                          # Defined: no __getattr__ call
x.job = 'coder'                 # Runs __setattr__ again
x.job                           # Defined: no __getattr__ call

# 4. The modern decorator syntax
class WithProperties:
    @property                   # Coding properties with decorators: ahead
    def age(self):              # On instance.age
    @age.setter
    def age(self, value):       # On instance.age = value

# __getattribute__ and Descriptors: Attribute Implementations
    # __getattribute__ intercepts every attribute read, while descriptors control access to specific attributes.
# 1. __getattribute__ versus __getattr__
class Person:
    def __getattribute__(self, name):
        print("Reading:", name)
        return object.__getattribute__(self, name)

p = Person()
p.age = 40
print(p.age)

    # A descriptor is an object that defines methods such as __get__() or __set__() to control access to a particular attribute.
class AgeDesc:
    def __get__(self, instance, owner): return 40
    def __set__(self, instance, value): instance._age = value

class WithDescriptors:
    age = AgeDesc()         # Assign descriptor instance

x = WithDescriptors()
x.age                   # Runs AgeDesc.__get__
x.age = 42              # Runs AgeDesc.__set__
x._age                  # Normal fetch: no AgeDesc call

# Static and Class Methods
    # static methods and class methods. Unlike ordinary instance methods,
    # they do not require a specific object instance in the usual way.
# 1. Ordinary instance method
class Student:
    def greet(self):
        print("Hello!")

s = Student()
s.greet()

# 2. Static method — @staticmethod
class Math:
    @staticmethod
    def add(a, b):
        return a + b
print(Math.add(3, 5))  # 8
m = Math()
print(m.add(4, 6))     # 10

# 3. Class method — @classmethod
class Student:
    school = "Central School"

    @classmethod
    def get_school(cls):
        return cls.school
print(Student.get_school())  # Central School

# Why the Special Methods?
    # sometimes a method needs to work with the class itself rather than with an individual object.
    # 1. The problem: tracking objects
    # 1. The problem: tracking objects
class Student:
    count = 0
    def __init__(self, name):
        self.name = name
        Student.count += 1
s1 = Student("Ali")
s2 = Student("Malika")
s3 = Student("John")
print(Student.count)  # 3

    # 2. Why not use an ordinary function outside the class?
def get_count():
    return Student.count

    # better organized:
class Student:
    count = 0
    def __init__(self, name):
        self.name = name
        Student.count += 1
    @staticmethod
    def get_count():
        return Student.count
s1 = Student("Ali")
s2 = Student("Malika")
print(Student.get_count())  # 2
print(s1.get_count())       # 2
    # 3. Why use a class method instead?
    # A class method receives the class itself through cls
class Student:
    count = 0
    @classmethod
    def get_count(cls):
        return cls.count
class GraduateStudent(Student):
    count = 10
print(Student.get_count())         # 0
print(GraduateStudent.get_count()) # 10

# Plain-Function Methods

# Static Method Alternatives

# Using Static and Class Methods

# Counting Instances with Static Methods

# Counting Instances with Class Methods
# Counting instances per class with class methods
# Decorators and Metaclasses
# Function Decorator Basics
# A First Look at User-Defined Function Decorators
# A First Look at Class Decorators and Metaclasses
# The super Function
# The super Basics
# The super Details
# A “magic” proxy
# Attribute-fetch algorithm
# Universal deployment
# Call-chain anchors
# Same argument lists
# Noncalls and operator overloading
# The super Wrap-Up
# Class Gotchas
# Changing Class Attributes Can Have Side Effects
# Changing Mutable Class Attributes Can Have Side Effects, Too
# Multiple Inheritance: Order Matters
# Scopes in Methods and Classes
# Miscellaneous Class Gotchas
# “Overwrapping-itis”


SyntaxError: invalid syntax (2247513062.py, line 267)